<a href="https://colab.research.google.com/github/RahebarShaik/Next-Word-Prediction-Using-LSTM/blob/main/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Data Collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

# Load the text data from the Gutenberg corpus
data = gutenberg.raw('shakespeare-hamlet.txt')

# Save the data to a text file
with open('hamlet.txt', 'w') as f:
    f.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [4]:
#Data Preprocessing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load the text data
with open('hamlet.txt', 'r') as f:
    text = f.read().lower()

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

In [5]:
total_words

4818

In [6]:
#Creating input sequences
inputsequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        inputsequences.append(n_gram_sequence)

In [7]:
inputsequences

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1, 687, 4, 45, 41, 1886, 1887, 1888, 1180],
 [1, 687, 4, 45, 41, 1886, 1887, 1888, 1180, 1889],
 [1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [1182, 63],
 [1182, 63, 408],
 [162, 377],
 [162, 377, 21],
 [162, 377, 21, 247],
 [162, 377, 21, 247, 882],
 [162, 377, 21, 247, 882, 18],
 [162, 377, 21, 247, 882, 18, 66],
 [162, 377, 21, 247, 882, 18, 66, 451],
 [224, 248],
 [224, 248, 1],
 [224, 248, 1, 30],
 [224, 248, 1, 30, 408],
 [407, 451],
 [25, 408],
 [6, 43],
 [6, 43, 62],
 [6, 43, 62, 1893],
 [6, 43, 62, 1893, 96],
 [6, 43, 62, 1893, 96, 18],
 [6, 43, 62, 1893, 96, 18, 566],
 [6, 43, 62, 1893, 96, 18, 566, 451],
 [71, 51],
 [71, 51, 1894],
 [71, 51, 1894, 567],
 [71, 51, 1894, 567, 378],
 [71, 51, 1894, 567, 378, 80],
 [71, 51, 1894, 567, 378

In [8]:
#Padding sequences
max_sequence_len = max([len(x) for x in inputsequences])
inputsequences = np.array(pad_sequences(inputsequences, maxlen=max_sequence_len, padding='pre'))
inputsequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [9]:
#Create predictors and label
import tensorflow as tf
x,y = inputsequences[:,:-1],inputsequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [10]:
#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [11]:
# Train the LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
model = Sequential()
model.add(Embedding(total_words,100))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100,))
model.add(Dense(total_words, activation='softmax'))
model.build(input_shape=(None, max_sequence_len-1))

model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])


In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 161, 100)       │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 161, 150)       │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 161, 150)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# from tensorflow.keras.callbacks import EarlyStopping
# es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5,restore_best_weights=True)

In [17]:
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test),verbose=1)

Epoch 1/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.0328 - loss: 6.5982 - val_accuracy: 0.0372 - val_loss: 6.7581
Epoch 2/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.0409 - loss: 6.3849 - val_accuracy: 0.0498 - val_loss: 6.8096
Epoch 3/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.0530 - loss: 6.2139 - val_accuracy: 0.0561 - val_loss: 6.8169
Epoch 4/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.0585 - loss: 6.0125 - val_accuracy: 0.0609 - val_loss: 6.8165
Epoch 5/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.0638 - loss: 5.8562 - val_accuracy: 0.0633 - val_loss: 6.8656
Epoch 6/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.0739 - loss: 5.7122 - val_accuracy: 0.0669 - val_loss: 6.9828
Epoch 7/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.0866 - loss: 5.5256 - val_accuracy: 0.0671 - val_loss: 7.0595
Epoch 8/50
696/696 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.0913 - loss: 5.3582 - 

In [18]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 12.1135
Test Accuracy: 0.0471


In [19]:
# Function to generate text
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [20]:
input_text = "to be or not to"
max_sequence_len = model.input_shape[1] + 1
print(predict_next_word(model, tokenizer, input_text, max_sequence_len))


tell


In [21]:
model.save('lstm_model.h5')

#Save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)